# Testing PL Switch Experiment Results for Fixed Cations and Varying Mobile Anions
## Overview

The aim of this set of simulation runs is to make the ongoing investigation into switch-based PL a little more realistic by keeping the cation distribution within the device fixed. 

To replicate the idea of a fixed positive ionic species within SIMsalabim, I will dope the device with donor atoms - matching the density I would desire for a mobile species - while setting the mobile cation density to 0. SIMsalabim will not accept 0 ion mobility, so this is the workaround one must take to simulate stationary ionic species. 

For this investigation, I will vary: 
1. The mobility of the mobile anions in the perovskite layer - l2.mu_anion
2. The density of both the mobile anions - l2.N_anion
3. The density of fixed cations - l2.N_D

## Experiment ID: "switch_PL_for_fixed_cations_varying_mobile_anions"
### Set Up

In [2]:
import os, sys
import numpy as np
import typing 
from numbers import Number
import itertools as it
from dataclasses import dataclass
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy.signal import find_peaks
import xarray as xr
try:
    import pySIMsalabim as sim
except ImportError: # add parent directory to sys.path if pySIMsalabim is not installed
    sys.path.append('../..')
    import pySIMsalabim as sim

In [3]:
experiment_id = "switch_PL_for_fixed_cations_varying_mobile_anions"
session_path = os.path.join("../../", "SIMsalabim/ZimT")

In [17]:
# Define fixed parameters that will remain unchanged for each run 
cation_density = 0

# Define varying parameters
anion_mobiliy_values = [1E-14, 5E-14, 1E-15]
ion_densitiy_values = np.geomspace(1.5E21, 1.5E23, 8)

# Here, I want density variations to be independent, and so use the Cartesian product of ion_density_values
parameter_combinations = np.array(list(it.product(ion_densitiy_values, ion_densitiy_values, anion_mobilities)))
fixed_cation_densities = parameter_combinations[:, 0]
mobile_anion_densities = parameter_combinations[:, 1]
anion_mobilities = parameter_combinations[:, 2]

# Pass all experiment parameters to a dictionary
experiment_parameters = {
    "l2.N_cation" : cation_density,
    "l2.mu_anion" : anion_mobilities,
    "l2.N_anion" : mobile_anion_densities,
    "l2.N_D" : fixed_cation_densities
}

output_files = {
    "tjFile" : "tj.dat",
    "varFile" : "varFile.dat"
}

In [18]:
# From the arguments and values, create a parameter .txt file alongside a .sh file containing all the various experiments to run
import uuid 
import hashlib

def get_UUID_from_string(s: str) -> uuid.UUID:
    '''Creates a UUID object based off a passed string'''
    hex_string = hashlib.md5(s.encode("UTF-8")).hexdigest()
    return uuid.UUID(hex=hex_string)

def generate_experiment_files(experiment_parameters: dict[str, float | np.ndarray], session_path: str, experiment_id: str, 
                              tVG_file: str = "tVG.txt", output_files: dict[str, str] = {"tjFile" : "tj.dat"}) -> str:
    '''Generates a bash script where each line corresponds to a specified experiment alongside a .txt file containing all passed parameters'''
    
    # Set up the header of the space seperated parameter file .txt
    parameter_file_string = "UUID "
    for parameter in experiment_parameters.keys(): 
        parameter_file_string += f"{parameter} "
    parameter_file_string = parameter_file_string.strip() + '\n'
    
    # Set up the experiment_zimt_commands file
    zimt_command = "./zimt"
    zimt_commands_file_string = ""
    
    # Set up the tVG command flag and argument
    tVG_file_command_line_argument = f" -tVGFile {tVG_file}"
    
    # Convert the passed experiment parameters to a dataFrame for easy iteration, and parse each row into the relevant files
    experiment_parameters_dataframe = pd.DataFrame(experiment_parameters)
    for _, run in experiment_parameters_dataframe.iterrows():
        run_parameters = ""
        run_command_line_arguments = ""
        
        for parameter, value in run.items():
            run_parameters += f"{value:.2e} "
            run_command_line_arguments += f" -{parameter} {value:.2e}"
        
        run_uuid = str(get_UUID_from_string(run_parameters))
        run_uuid_savefile_tag = f"_{run_uuid}"
        
        for savefile_flag, savefile_name in output_files.items():
            savefile_name_base, savefile_name_extension = os.path.splitext(savefile_name)
            savefile_name = savefile_name_base + f"__{experiment_id}_" + run_uuid_savefile_tag + savefile_name_extension
            
            run_command_line_arguments += f" -{savefile_flag} {savefile_name}"
            
            
        parameter_file_string += run_uuid + " " + run_parameters.strip() + "\n"
        zimt_commands_file_string += zimt_command + tVG_file_command_line_argument + run_command_line_arguments + "\n"
        
    # Save experiment parameters
    parameter_file_name = f"experiment__{experiment_id}__parameters.txt"
    parameter_file_path = os.path.join(session_path, parameter_file_name)
    with open(parameter_file_path, 'w') as file:
        file.write(parameter_file_string)
        
    # Save zimt commands 
    zimt_commands_file_name = f"experiment__{experiment_id}__zimt_commands.sh"
    zimt_commands_file_path = os.path.join(session_path, zimt_commands_file_name)
    with open(zimt_commands_file_path, 'w') as file:
        file.write(zimt_commands_file_string)
        
    return parameter_file_path

In [19]:
parameter_file_path = generate_experiment_files(experiment_parameters, session_path, experiment_id, tVG_file="tVG_high_res.dat", output_files=output_files)

### Experiment Execution
Run the generated experiment bash script commands with 

```~$ parallel :::: experiment__switch_PL_for_fixed_cations_varying_mobile_anions__zimt_commands.sh```

Test the generated experiment bash script commands with 

```~$ parallel --dry-run :::: experiment__switch_PL_for_fixed_cations_varying_mobile_anions__zimt_commands.sh```